# Silver province reporting-area population

## Purpose

Aggregate authoritative Table C population from accepted barangay rows to the D-55 reporting-area contract without subtotal or independent-city duplication.

## Grain

One governed reporting area per reference year, authoritative source, selected reconciliation run, and deterministic transformation run.


## 1. Gate current geography and population reconciliation


In [0]:
%sql
USE CATALOG `buildabida-capstone`;
CREATE SCHEMA IF NOT EXISTS `02-silver`;

DECLARE OR REPLACE VARIABLE population_rule_version STRING
    DEFAULT 'silver-province-reporting-area-population-v1';
DECLARE OR REPLACE VARIABLE ncr_region_code STRING DEFAULT '1300000000';

SELECT ASSERT_TRUE(
    COUNT(*) > 0
    AND COUNT(DISTINCT run_id) = 1
    AND COUNT(*) = COUNT(DISTINCT reporting_area_key)
    AND COUNT_IF(reporting_area_key = 0) = 1,
    'STOP: Silver reporting-area reference is empty, mixed-run, or duplicated'
)
FROM `02-silver`.silver_province_reporting_area;

SELECT ASSERT_TRUE(
    COUNT(*) > 0
    AND COUNT(DISTINCT run_id) = 1
    AND COUNT(DISTINCT source_snapshot_id) = 1
    AND COUNT(DISTINCT psgc_source_snapshot_id) = 1,
    'STOP: population reconciliation is empty or mixed-run'
)
FROM `02-silver`.silver_population_place_reconciliation;

SELECT ASSERT_TRUE(
    COUNT_IF(
        row_disposition = 'ACCEPTED_PRIMARY'
        AND (population_count <= 0 OR matched_place_type <> 'BARANGAY')
    ) = 0
    AND COUNT_IF(row_disposition = 'ACCEPTED_PRIMARY') > 0,
    'STOP: accepted population rows must be positive matched barangays'
)
FROM `02-silver`.silver_population_place_reconciliation;


## 2. Resolve each accepted barangay once

NCR barangays roll to the one NCR reporting area. A barangay with an official province parent rolls to that province. A barangay in a city without a province parent rolls to that independent-city reporting area. Any accepted row that cannot resolve stays under key 0 and remains visible.


In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW reporting_area_population_lineage AS
SELECT
    MAX(reference_year) AS reference_year,
    MAX(population_reference_date) AS population_reference_date,
    MAX(source_name) AS source_name,
    MAX(source_snapshot_id) AS source_snapshot_id,
    MAX(source_ingest_run_id) AS source_ingest_run_id,
    MAX(psgc_source_snapshot_id) AS psgc_source_snapshot_id,
    MAX(psgc_source_ingest_run_id) AS psgc_source_ingest_run_id,
    MAX(psgc_version) AS psgc_version,
    MAX(alias_version) AS alias_version,
    MAX(row_role_exception_version) AS row_role_exception_version,
    MAX(run_id) AS population_reconciliation_run_id,
    MAX(source_load_ts) AS source_load_ts
FROM `02-silver`.silver_population_place_reconciliation;

CREATE OR REPLACE TEMPORARY VIEW accepted_barangay_reporting_area AS
SELECT
    population.source_row_identity,
    population.population_count,
    CASE
        WHEN population.matched_region_code = ncr_region_code THEN ncr_region_code
        WHEN place.province_code IS NOT NULL THEN place.province_code
        WHEN place.place_type = 'BARANGAY'
         AND locality.place_type = 'CITY'
         AND locality.province_code IS NULL
            THEN place.city_municipality_code
    END AS psgc_reporting_area_code
FROM `02-silver`.silver_population_place_reconciliation AS population
INNER JOIN `02-silver`.silver_psgc_place AS place
    ON population.matched_psgc_code = place.psgc_code
   AND population.psgc_source_snapshot_id = place.source_snapshot_id
LEFT JOIN `02-silver`.silver_psgc_place AS locality
    ON place.city_municipality_code = locality.psgc_code
   AND place.source_snapshot_id = locality.source_snapshot_id
WHERE population.row_disposition = 'ACCEPTED_PRIMARY'
  AND population.matched_place_type = 'BARANGAY'
  AND population.population_count > 0;

CREATE OR REPLACE TEMPORARY VIEW accepted_barangay_reporting_area_keyed AS
SELECT
    accepted.source_row_identity,
    accepted.population_count,
    COALESCE(area.reporting_area_key, CAST(0 AS BIGINT)) AS reporting_area_key
FROM accepted_barangay_reporting_area AS accepted
LEFT JOIN `02-silver`.silver_province_reporting_area AS area
    ON accepted.psgc_reporting_area_code = area.psgc_reporting_area_code;

SELECT ASSERT_TRUE(
    COUNT(*) = COUNT(DISTINCT source_row_identity)
    AND COUNT(*) = (
        SELECT COUNT(*)
        FROM `02-silver`.silver_population_place_reconciliation
        WHERE row_disposition = 'ACCEPTED_PRIMARY'
          AND matched_place_type = 'BARANGAY'
          AND population_count > 0
    ),
    'STOP: an accepted barangay disappeared or entered more than one reporting area'
)
FROM accepted_barangay_reporting_area_keyed;


## 3. Publish and reconcile the population table


In [0]:
%sql
CREATE OR REPLACE TABLE `02-silver`.silver_province_reporting_area_population
USING DELTA
COMMENT 'Authoritative Table C population aggregated from accepted barangays to D-55 province reporting areas without subtotal or independent-city duplication.'
AS
WITH population_by_area AS (
    SELECT
        reporting_area_key,
        SUM(population_count) AS population_count,
        COUNT(*) AS source_row_count
    FROM accepted_barangay_reporting_area_keyed
    GROUP BY reporting_area_key
),
excluded AS (
    SELECT
        SUM(CASE
            WHEN row_role IN ('APPROVED_UPPERCASE_BARANGAY', 'NORMAL_BARANGAY_CANDIDATE')
             AND population_count > 0 THEN population_count
        END)
            AS excluded_population_count,
        COUNT(*) AS excluded_source_row_count
    FROM `02-silver`.silver_population_place_reconciliation
    WHERE row_disposition <> 'ACCEPTED_PRIMARY'
)
SELECT
    area.reporting_area_key,
    area.psgc_reporting_area_code,
    area.reporting_area_name,
    area.reporting_area_type,
    area.parent_region_code,
    lineage.reference_year,
    lineage.population_reference_date,
    population.population_count,
    lineage.source_name,
    TRUE AS is_primary_source,
    CASE
        WHEN population.population_count IS NULL THEN 'MISSING_PRIMARY_POPULATION'
        WHEN area.reporting_area_key = 0 THEN 'UNRESOLVED_ACCEPTED_POPULATION'
        ELSE 'MATCHED_PRIMARY_POPULATION'
    END AS population_match_status,
    COALESCE(population.source_row_count, 0) AS source_row_count,
    CASE WHEN area.reporting_area_key = 0 THEN excluded.excluded_population_count END
        AS excluded_population_count,
    CASE WHEN area.reporting_area_key = 0 THEN excluded.excluded_source_row_count END
        AS excluded_source_row_count,
    lineage.source_snapshot_id,
    lineage.source_ingest_run_id,
    lineage.psgc_source_snapshot_id,
    lineage.psgc_source_ingest_run_id,
    lineage.psgc_version,
    lineage.alias_version,
    lineage.row_role_exception_version,
    area.run_id AS reporting_area_run_id,
    lineage.population_reconciliation_run_id,
    population_rule_version AS transformation_rule_version,
    SHA2(CONCAT_WS(
        '||', lineage.population_reconciliation_run_id, area.run_id,
        lineage.alias_version, lineage.row_role_exception_version,
        population_rule_version
    ), 256) AS run_id,
    GREATEST(lineage.source_load_ts, area.source_load_ts) AS source_load_ts
FROM `02-silver`.silver_province_reporting_area AS area
CROSS JOIN reporting_area_population_lineage AS lineage
CROSS JOIN excluded
LEFT JOIN population_by_area AS population
    ON area.reporting_area_key = population.reporting_area_key;

SELECT ASSERT_TRUE(
    COUNT(*) = COUNT(DISTINCT NAMED_STRUCT(
        'area', reporting_area_key, 'year', reference_year, 'source', source_name
    ))
    AND SUM(population_count) <=> (
        SELECT SUM(population_count) FROM accepted_barangay_reporting_area_keyed
    ),
    'STOP: reporting-area population grain or accepted-primary total does not reconcile'
)
FROM `02-silver`.silver_province_reporting_area_population;


# Summary

Only positive `ACCEPTED_PRIMARY` barangays contribute. Every accepted row enters one reporting area, while excluded source population is disclosed separately. Missing area population stays NULL. Databricks runtime evidence is still required.
